# TD4 — Feature engineering EEG pour l'estimation de la charge cognitive

## Contexte

Ce TD s'inscrit dans le projet fil rouge basé sur le papier :

**Multimodal Brain-Computer Interface for In-Vehicle Driver Cognitive Load Measurement: Dataset and Baselines**.

Le papier introduit le dataset **CL-Drive**, dans lequel des signaux EEG, ECG, EDA et Gaze sont enregistrés pendant une tâche de conduite simulée. Les scores de charge cognitive sont collectés toutes les **10 secondes**. Après prétraitement, les signaux sont segmentés en fenêtres de 10 s, puis transformés en caractéristiques numériques pour entraîner des modèles de classification.

Dans ce TD, on se concentre uniquement sur les **features EEG**.

---

## Objectifs pédagogiques

À la fin du TD, vous devez être capables de :

1. expliquer pourquoi on transforme un signal EEG en vecteur de caractéristiques ;
2. distinguer les features temporelles, fréquentielles et non linéaires ;
3. expliquer le principe de la densité spectrale de puissance, ou PSD ;
4. calculer des puissances par bande EEG ;
5. interpréter l'entropie spectrale ;
6. expliquer les paramètres de Hjorth ;
7. comprendre le principe de la complexité de Lempel-Ziv ;
8. comprendre l'idée de la dimension fractale de Higuchi ;
9. structurer une fonction complète d'extraction de features EEG ;
10. préparer une matrice de features pour la classification.

---

## Features EEG ciblées

Le papier regroupe les features EEG suivantes :

| Famille | Features |
|---|---|
| PSD | puissance absolue, moyenne, maximale, minimale et médiane |
| Entropie spectrale | entropie calculée à partir de la PSD normalisée |
| Hjorth | mobility et complexity |
| Lempel-Ziv | complexité d'une séquence binarisée |
| Higuchi | dimension fractale |
| Statistiques temporelles | moyenne, minimum, maximum, médiane, variance, écart-type |

Dans ce TD, on adopte une version complète :

- PSD calculée dans les cinq bandes EEG : delta, theta, alpha, beta, gamma ;
- entropie spectrale calculée dans les cinq bandes ;
- features non linéaires calculées sur le segment temporel ;
- statistiques temporelles calculées sur le segment.

On obtient donc :

$$
5 \text{ bandes} \times 5 \text{ descripteurs PSD} = 25
$$

$$
5 \text{ entropies spectrales} = 5
$$

$$
2 \text{ Hjorth} + 1 \text{ Lempel-Ziv} + 1 \text{ Higuchi} + 6 \text{ statistiques} = 10
$$

Soit au total :

$$
25 + 5 + 10 = 40 \text{ features par canal EEG}
$$

## Questions de compréhension

### Question 1

Pourquoi ne donne-t-on pas directement le signal EEG brut à un classifieur classique comme LDA, SVM ou Random Forest ?

### Réponse 

Le signal EEG brut est très long, bruité et non stationnaire : le fournir tel quel à un classifieur classique conduit souvent à du surapprentissage, de mauvaises performances et à des coûts de calcul élevés.
L'extraction de features (puissances par bande, entropie, paramètres de Hjorth, complexités, statistiques temporelles) permet de : réduire la dimension, stabiliser l'estimation en améliorant le rapport signal/bruit, fournir des vecteurs de taille fixe requis par LDA/SVM/RandomForest, et mettre en évidence l'information physiologique pertinente.
Sans ces transformations, il faudrait énormément de données et d'ingénierie pour que les classifieurs apprennent des motifs robustes dans le signal brut.

### Question 2

Pourquoi les features fréquentielles sont-elles particulièrement importantes en EEG ?

### Réponse 

Les features fréquentielles sont cruciales car l'activité cérébrale se manifeste souvent par des rythmes (delta, theta, alpha, beta, gamma) liés à des états cognitifs et comportementaux distincts.
Les puissances par bande et l'entropie spectrale capturent comment l'énergie du signal se répartit en fréquence, ce qui est directement relié à l'attention, la charge cognitive, la relaxation, etc.
De plus, les caractéristiques fréquentielles sont en général plus robustes aux décalages de phase et permettent d'isoler des phénomènes physiologiques difficiles à repérer dans le domaine temporel seul.

### Question 3

Pourquoi faut-il calculer les features séparément sur chaque canal EEG ?

### Réponse 

Chaque électrode enregistre l'activité d'une région corticale différente ; les patterns utiles peuvent donc être localisés spatialement.
Calculer les features par canal conserve cette information spatiale, permet de gérer des canaux corrompus ou bruités individuellement, et facilite l'interprétabilité (par exemple, quelles régions montrent une augmentation d'alpha).
Enfin, concaténer des features par canal permet aux modèles de tirer parti des corrélations spatiales tout en gardant une traçabilité par capteur.

## 1. Bandes fréquentielles EEG

Les signaux EEG sont souvent analysés par bandes de fréquence.

| Bande | Intervalle utilisé dans ce TD | Interprétation générale |
|---|---:|---|
| Delta | 0.5–4 Hz | activité lente |
| Theta | 4–8 Hz | attention, mémoire de travail, somnolence selon contexte |
| Alpha | 8–12 Hz | relaxation, inhibition, yeux fermés |
| Beta | 12–30 Hz | activité mentale, attention, activité motrice |
| Gamma | 30–75 Hz | activité rapide, intégration, mais sensible aux artefacts musculaires |

Dans le papier, la bande gamma va jusqu'à 75 Hz. 

### Question 4

Pourquoi peut-on limiter la bande gamma à 45 Hz dans certaines implémentations ?

### Réponse 

On limite parfois la bande gamma à ~45 Hz pour réduire la contamination par le bruit non neuronale et les artefacts (EMG, micro-mouvements, interférences secteur autour de 50/60 Hz).
Les signaux au‑dessus de 45 Hz sont souvent dominés par des activités musculaires et par le bruit d'alimentation, ce qui réduit la spécificité des mesures EEG si le prétraitement n'est pas strict.
De plus, limiter la bande à 45 Hz diminue les exigences d'échantillonnage et de filtrage (moins de puissance de calcul et moins d'artefacts de repliement), ce qui est utile en applications embarquées ou quand l'acquisition/amplificateur n'est pas calibré pour hautes fréquences.
Toutefois, si l'on veut étudier le high-gamma (>45 Hz), il faut une chaîne d'acquisition et des étapes de nettoyage (référence, suppression EMG, filtres anti-aliasing, échantillonnage plus élevé) adaptées.

## 2. Méthodologie d'implémentation 

Dans ce TD, l'objectif est de comprendre puis implémenter les fonctions essentielles.

Pour chaque fonction, vous aurez :

- une explication théorique ;
- l'algorithme ;
- les fonctions Python recommandées ;
- les paramètres importants ;
- des questions de vérification.

Les bibliothèques utiles sont :

| Objectif | Bibliothèque | Fonctions utiles |
|---|---|---|
| Tableaux numériques | NumPy | `np.array`, `np.mean`, `np.var`, `np.diff`, `np.median` |
| Données tabulaires | pandas | `pd.DataFrame`, `pd.read_csv`, `to_csv` |
| PSD | scipy.signal | `welch` |
| Entropie | scipy.stats | `entropy` |
| Visualisation | matplotlib | `plt.plot`, `plt.semilogy`, `plt.bar` |

### Travail à réaliser

Vous devez construire progressivement les fonctions suivantes :

1. `compute_psd_band_features(signal, fs)` ;
2. `compute_spectral_entropy_bands(signal, fs)` ;
3. `compute_hjorth(signal)` ;
4. `lempel_ziv_complexity(signal)` ;
5. `higuchi_fd(signal, kmax=10)` ;
6. `compute_raw_features(signal)` ;
7. `extract_eeg_features(signal, fs)` ;
8. une boucle permettant d'extraire les features sur des fenêtres de 10 secondes.

In [5]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import welch
from scipy.stats import entropy

FS = 256
WINDOW_SEC = 10
WINDOW_SAMPLES = FS * WINDOW_SEC

EEG_BANDS = {
    "delta": (0.5, 4),
    "theta": (4, 8),
    "alpha": (8, 12),
    "beta": (12, 31),
    "gamma": (31, 75),
}

print("Nombre d'échantillons par fenêtre :", WINDOW_SAMPLES)
print("Bandes EEG :", EEG_BANDS)

Nombre d'échantillons par fenêtre : 2560
Bandes EEG : {'delta': (0.5, 4), 'theta': (4, 8), 'alpha': (8, 12), 'beta': (12, 31), 'gamma': (31, 75)}


## 3. Exemples pédagogiques sur signaux connus

Avant d'appliquer les features à l'EEG réel, on commence par des signaux connus.

On va comparer :

1. une sinusoïde à 2 Hz, principalement dans la bande delta ;
2. une sinusoïde à 10 Hz, principalement dans la bande alpha ;
3. une sinusoïde à 20 Hz, principalement dans la bande beta ;
4. un bruit blanc, dont l'énergie est plus répartie ;
5. un mélange de plusieurs composantes.

### Objectif pédagogique

Vérifier que les features donnent des résultats cohérents :

- une sinusoïde pure doit avoir une PSD concentrée ;
- le bruit doit avoir une entropie spectrale plus élevée ;
- une sinusoïde alpha doit avoir une puissance alpha dominante.

In [6]:
# Cellule d'aide : génération de signaux synthétiques pour les tests pédagogiques.
# Vous pouvez l'utiliser pour tester vos futures fonctions de features.

def generate_synthetic_signals(fs=FS, duration=10, random_state=0):
    rng = np.random.default_rng(random_state)
    t = np.arange(0, duration, 1/fs)
    signals = {
        "delta_2Hz": np.sin(2*np.pi*2*t),
        "alpha_10Hz": np.sin(2*np.pi*10*t),
        "beta_20Hz": np.sin(2*np.pi*20*t),
        "white_noise": rng.normal(0, 1, size=len(t)),
        "mixed": 0.8*np.sin(2*np.pi*6*t) + 0.5*np.sin(2*np.pi*10*t) + 0.2*rng.normal(0, 1, size=len(t)),
    }
    return t, signals

t, synthetic_signals = generate_synthetic_signals()

## 4. Feature PSD : densité spectrale de puissance

### Principe théorique

La **PSD**, ou densité spectrale de puissance, indique comment la puissance du signal est répartie selon la fréquence.

Pour un segment EEG $x[n]$, on estime la PSD avec la méthode de Welch. L'idée est de :

1. découper le signal en sous-fenêtres ;
2. calculer un spectre sur chaque sous-fenêtre ;
3. moyenner les spectres pour obtenir une estimation plus stable.

En Python, on utilise généralement :

```python
freqs, psd = scipy.signal.welch(signal, fs=fs, nperseg=...)
```

### Paramètres recommandés

- `fs=256` pour CL-Drive ;
- `nperseg=fs*2` pour une fenêtre Welch de 2 secondes ;
- si le segment est court, prendre `nperseg=min(len(signal), fs*2)` ;
- sélectionner ensuite les fréquences appartenant à une bande donnée à l’aide d’un masque booléen (utiliser le vecteur `freqs` retourné `scipy.signal.welch`).

### Features PSD pour chaque bande

Pour chaque bande EEG, calculer :

1. puissance absolue : somme de la PSD dans la bande ;
2. puissance moyenne ;
3. puissance maximale ;
4. puissance minimale ;
5. puissance médiane.

### Algorithme

1. calculer `freqs, psd` avec Welch ;

Pour chaque bande $[f_{min}, f_{max}]$ :

   
2. sélectionner les indices tels que $f_{min} \le f < f_{max}$ ;
3. extraire `band_psd` ;
4. calculer `sum`, `mean`, `max`, `min`, `median` ;
5. stocker les résultats dans un dictionnaire.

### Questions

1. Quelle bande doit dominer pour un signal sinusoïdal à 10 Hz ?
2. Pourquoi la PSD est-elle plus stable avec Welch qu'avec un simple spectre FFT ?
3. Que se passe-t-il si la bande sélectionnée ne contient aucune fréquence ?

### Réponses 

1. Pour un signal sinusoïdal pur à 10 Hz, la bande `alpha` (8–12 Hz) doit dominer : la PSD montre un pic net autour de 10 Hz.
2. Welch est plus stable car il moyenne des périodogrammes calculés sur plusieurs sous‑fenêtres (fenêtrage + recouvrement), réduisant la variance de l'estimation et atténuant les fuites spectrales d'un unique FFT sur tout le segment.
3. Si la bande sélectionnée ne contient aucune énergie, les mesures PSD seront proches de zéro et certaines statistiques (somme, moyenne, max) seront nulles ; l'entropie peut être indéfinie si on divise par zéro. Il faut gérer ce cas (retourner 0/NaN, ajouter un petit `epsilon` ou marquer la feature comme absente) pour éviter des erreurs ou biais lors de l'entraînement.

In [7]:
# À faire dans votre notebook de travail : implémenter compute_welch_psd() et compute_psd_band_features().
# Indication : utiliser scipy.signal.welch, puis sélectionner chaque bande avec un masque fréquentiel.
from scipy.signal import welch
import numpy as np

def compute_welch_psd(signal, fs):
    x = np.asarray(signal)
    freqs, psd = welch(x, fs=fs, nperseg=min(len(x), fs*2))
    return freqs, psd

def compute_psd_band_features(signal, fs):
    freqs, psd = compute_welch_psd(signal, fs)
    features = {}
    for band, (fmin, fmax) in EEG_BANDS.items():
        mask = (freqs >= fmin) & (freqs < fmax)
        band_psd = psd[mask]
        if band_psd.size == 0:
            features[f'{band}_psd_sum'] = 0.0
            features[f'{band}_psd_mean'] = 0.0
            features[f'{band}_psd_max'] = 0.0
            features[f'{band}_psd_min'] = 0.0
            features[f'{band}_psd_median'] = 0.0
        else:
            features[f'{band}_psd_sum'] = float(band_psd.sum())
            features[f'{band}_psd_mean'] = float(band_psd.mean())
            features[f'{band}_psd_max'] = float(band_psd.max())
            features[f'{band}_psd_min'] = float(band_psd.min())
            features[f'{band}_psd_median'] = float(np.median(band_psd))
    return features

# Exemple d'utilisation correct :
alpha_features = compute_psd_band_features(synthetic_signals['alpha_10Hz'], FS)

## 5. Entropie spectrale

### Principe théorique

L'entropie spectrale mesure la dispersion de l'énergie dans le domaine fréquentiel.

On calcule d'abord la PSD dans une bande, puis on la normalise pour obtenir une distribution :

$$
p_i = \frac{PSD_i}{\sum_j PSD_j}
$$

L'entropie de Shannon est ensuite :

$$
H = - \sum_i p_i \log(p_i)
$$


### Fonction Python utile

```python
from scipy.stats import entropy
```

`entropy(p)` calcule directement l'entropie de Shannon si `p` est une distribution normalisée.

### Questions

1. Pourquoi faut-il normaliser la PSD avant de calculer l'entropie ?
2. Quel signal devrait avoir l'entropie spectrale la plus élevée : une sinusoïde pure ou un bruit blanc ?
3. Pourquoi l'entropie spectrale peut-elle être utile pour caractériser la complexité d'un EEG ?

### Réponses 

**1.** L'entropie de Shannon est définie sur une **distribution de probabilité** : les valeurs $p_i$ doivent être positives et sommer à 1. La PSD brute est exprimée en V²/Hz et sa somme dépend de l'amplitude et de la durée du signal — deux grandeurs sans lien avec la répartition fréquentielle. Normaliser par $\sum_j PSD_j$ convertit la PSD en distribution relative, ce qui rend l'entropie comparable entre canaux, entre sujets et entre fenêtres, indépendamment du niveau absolu du signal.

**2.** Le **bruit blanc** devrait avoir l'entropie spectrale la plus élevée. Son énergie est uniformément répartie sur toutes les fréquences, ce qui correspond à une distribution $p_i$ quasiment uniforme — état de désordre maximal. Une sinusoïde pure concentre toute son énergie sur un seul bin fréquentiel : la distribution est très piquée, presque toute la masse sur une seule valeur, donc l'entropie est proche de zéro.

**3.** En EEG, un signal de faible charge cognitive tend à être dominé par un ou deux rythmes (ex. alpha), ce qui donne une entropie spectrale faible (énergie concentrée). Une charge cognitive élevée active simultanément plusieurs bandes (theta, beta, gamma), dispersant l'énergie et augmentant l'entropie. L'entropie spectrale est donc un **indicateur de complexité fonctionnelle** : elle capture en un seul scalaire si l'activité cérébrale est focalisée ou multi-bandes, sans avoir à interpréter chaque bande séparément.

In [8]:
# À vous d'implémenter compute_spectral_entropy_bands().
import scipy as scp
from scipy.stats import entropy

def compute_specetral_entropy_bands(signal, fs):

    f, Pxx = scp.signal.welch(signal, fs)

    features = {}
    for band_name, (fmin, fmax) in EEG_BANDS.items():
        mask = (f >= fmin) & (f < fmax)
        band_psd = Pxx[mask]

        total = band_psd.sum()
        if total == 0:
            features[f"spectral_entropy_{band_name}"] = 0.0
        else:
            p = band_psd / total  # normalisation : p_i = PSD_i / sum(PSD_j)
            features[f"spectral_entropy_{band_name}"] = float(entropy(p))

    return features

## 6. Paramètres de Hjorth : mobility et complexity

Les paramètres de Hjorth sont des descripteurs temporels utilisés pour caractériser la dynamique d'un signal.

### Mobility

La mobility mesure approximativement la fréquence moyenne du signal :

$$
Mobility(x) = \sqrt{\frac{Var(\Delta x)}{Var(x)}}
$$

où $\Delta x$ est la dérivée discrète du signal, généralement calculée avec `np.diff(x)`.

### Complexity

La complexity mesure la variation de la mobility entre le signal et sa dérivée :

$$
Complexity(x) = \frac{Mobility(\Delta x)}{Mobility(x)}
$$


### Questions

1. Que vaut approximativement la variance d'un signal constant ?
2. Pourquoi faut-il gérer le cas où `Var(x)=0` ?
3. Entre une sinusoïde lisse et un bruit blanc, lequel devrait avoir une complexity plus élevée ?

### Réponses

1. La variance d'un signal constant vaut exactement **0** : tous les échantillons sont identiques, donc l'écart à la moyenne est nul.

2. Si `Var(x) = 0`, on divise par zéro dans le calcul de la mobility. Il faut donc intercepter ce cas et retourner `0` (ou `NaN`) pour éviter une erreur runtime et signaler que le signal est plat (canal mort ou saturé).

3. Le **bruit blanc** devrait avoir une complexity bien plus élevée. Une sinusoïde lisse a une dérivée qui oscille à la même fréquence, donc sa mobility et celle de sa dérivée sont proches, et leur rapport (la complexity) est proche de 1. Un bruit blanc a une dérivée encore plus irrégulière que lui-même, ce qui fait monter la complexity.

In [9]:
import numpy as np

def _mobility(x):
    """Mobility de Hjorth : sqrt(Var(diff(x)) / Var(x)). Retourne 0 si Var(x)=0."""
    var_x = np.var(x)
    if var_x == 0:
        return 0.0
    return np.sqrt(np.var(np.diff(x)) / var_x)

def compute_hjorth(signal):
    """
    Calcule les paramètres de Hjorth mobility et complexity.

    mobility  = sqrt( Var(diff(x))  / Var(x) )
    complexity = mobility(diff(x)) / mobility(x)

    Retourne un dict {"hjorth_mobility": ..., "hjorth_complexity": ...}.
    """
    x = np.array(signal, dtype=np.float64)

    mob = _mobility(x)
    mob_diff = _mobility(np.diff(x))

    complexity = (mob_diff / mob) if mob != 0 else 0.0

    return {
        "hjorth_mobility":   mob,
        "hjorth_complexity": complexity,
    }


# --- Vérification sur signaux synthétiques ---
FS = 256
duration = 10
t = np.arange(0, duration, 1 / FS)
rng = np.random.default_rng(0)

signals = {
    "sinus_lisse (10 Hz)": np.sin(2 * np.pi * 10 * t),
    "bruit_blanc":         rng.normal(0, 1, size=len(t)),
    "signal_constant":     np.ones(len(t)),
    "mixed":               0.8 * np.sin(2 * np.pi * 6 * t)
                           + 0.5 * np.sin(2 * np.pi * 10 * t)
                           + 0.2 * rng.normal(0, 1, size=len(t)),
}

print(f"{'Signal':<25}  {'Mobility':>12}  {'Complexity':>12}")
print("-" * 52)
for name, sig in signals.items():
    feat = compute_hjorth(sig)
    print(f"{name:<25}  {feat['hjorth_mobility']:>12.4f}  {feat['hjorth_complexity']:>12.4f}")

# --- Interprétation attendue ---
print("\nInterprétation attendue :")
print("  - sinus lisse    : mobility faible, complexity ≈ 1")
print("  - bruit blanc    : mobility élevée, complexity >> 1")
print("  - signal constant: mobility = 0, complexity = 0 (cas dégénéré géré)")

Signal                         Mobility    Complexity
----------------------------------------------------
sinus_lisse (10 Hz)              0.2448        1.0007
bruit_blanc                      1.4189        1.2207
signal_constant                  0.0000        0.0000
mixed                            0.4403        3.6457

Interprétation attendue :
  - sinus lisse    : mobility faible, complexity ≈ 1
  - bruit blanc    : mobility élevée, complexity >> 1
  - signal constant: mobility = 0, complexity = 0 (cas dégénéré géré)


## 7. Complexité de Lempel-Ziv

### Principe théorique

La complexité de Lempel-Ziv mesure le nombre de motifs nouveaux rencontrés dans une séquence.

Comme l'algorithme s'applique à une séquence symbolique, un signal EEG réel doit d'abord être transformé en séquence binaire.

Une méthode simple consiste à binariser le signal par rapport à sa médiane :

$$
b[n] =
\begin{cases}
1, & x[n] > median(x) \\
0, & x[n] \le median(x)
\end{cases}
$$


### Paramètres importants

- seuil de binarisation : médiane ou moyenne ;
- normalisation de la complexité pour comparer des segments de même ou de différente longueur ;
- gestion des segments constants.

### Questions

1. Pourquoi faut-il binariser le signal avant de calculer Lempel-Ziv ?
2. Pourquoi la médiane est-elle un seuil intéressant ?
3. Quel signal devrait avoir une complexité plus élevée : une sinusoïde pure ou un bruit blanc ?
4. Pourquoi normaliser la complexité par la longueur de la séquence ?

### Réponses 

1. L'algorithme Lempel‑Ziv opère sur des séquences symboliques (bits ou symboles discrets). Binariser transforme le signal continu en une suite de symboles finie, permettant de compter l'apparition de nouveaux motifs et d'estimer la complexité algorithmique de la séquence.
2. La médiane est robuste aux valeurs extrêmes et garantit une proportion approximativement égale de 0 et 1 pour des signaux symétriques, ce qui évite les déséquilibres de symboles (peu de transitions) et rend la mesure de complexité plus informative et stable entre segments.
3. Le bruit blanc a une complexité bien plus élevée. Une sinusoïde pure produit une séquence très régulière (peu de nouveaux motifs), tandis que le bruit génère beaucoup de motifs imprévisibles et donc une valeur Lempel‑Ziv plus grande.
4. La complexité brute croît avec la longueur de la séquence; normaliser (par exemple diviser par n/log n ou par la longueur) permet de comparer segments de durées différentes et d'obtenir une mesure relative indépendante de la taille, facilitant les comparaisons entre fenêtres ou sujets.

In [10]:
import numpy as np

def lempel_ziv_complexity(signal):
    """
    Complexité de Lempel-Ziv (LZ76) normalisée par n / log2(n).

    Étapes :
      1. Binarisation par la médiane : 1 si x[n] > médiane, 0 sinon.
      2. Parcours gauche-droite : on étend le motif courant jusqu'à
         trouver un sous-mot non encore rencontré, puis on incrémente c.
      3. Normalisation par n / log2(n) pour comparer des segments
         de longueurs différentes.
    """
    x = np.asarray(signal, dtype=np.float64)
    seq = tuple((x > np.median(x)).astype(int).tolist())
    n = len(seq)

    if n <= 1:
        return 0.0

    seen = set()
    c = 0
    i = 0

    while i < n:
        j = i + 1
        while j <= n and seq[i:j] in seen:
            j += 1
        seen.add(seq[i:j])
        c += 1
        i = j

    return float(c / (n / np.log2(n)))


# --- Vérification sur signaux synthétiques ---
FS = 256
t = np.arange(0, 10, 1 / FS)
rng = np.random.default_rng(0)

signals_lz = {
    "sinus pur (10 Hz)": np.sin(2 * np.pi * 10 * t),
    "bruit blanc":        rng.normal(0, 1, size=len(t)),
    "signal constant":    np.ones(len(t)),
}

print(f"{'Signal':<22}  {'LZ complexity':>14}")
print("-" * 40)
for name, sig in signals_lz.items():
    print(f"{name:<22}  {lempel_ziv_complexity(sig):>14.4f}")

print("\nInterprétation attendue :")
print("  - sinus pur     : LZ faible  (séquence très régulière)")
print("  - bruit blanc   : LZ élevée  (beaucoup de nouveaux motifs)")
print("  - signal constant: LZ ≈ 0   (un seul symbole)")

Signal                   LZ complexity
----------------------------------------
sinus pur (10 Hz)               0.9641
bruit blanc                     1.6452
signal constant                 0.3184

Interprétation attendue :
  - sinus pur     : LZ faible  (séquence très régulière)
  - bruit blanc   : LZ élevée  (beaucoup de nouveaux motifs)
  - signal constant: LZ ≈ 0   (un seul symbole)


## 8. Dimension fractale de Higuchi

### Principe théorique

La dimension fractale de Higuchi cherche à mesurer la complexité géométrique d'un signal temporel.

Un signal très lisse ressemble davantage à une courbe régulière. Un signal très irrégulier ou bruité présente une trajectoire plus complexe.

L'algorithme de Higuchi construit plusieurs sous-séquences avec différents pas $k$, mesure leur longueur moyenne $L(k)$, puis estime une pente dans un espace logarithmique.

### Paramètre important

- `kmax` : pas maximal testé.

Pour un segment de 10 secondes à 256 Hz, une valeur pédagogique simple est :

```python
kmax = 10
```

Une valeur trop faible peut donner une estimation instable ; une valeur trop élevée augmente le coût de calcul.

### Questions

1. Que cherche à mesurer la dimension fractale de Higuchi ?
2. Pourquoi un signal bruité peut-il avoir une dimension fractale plus élevée qu'une sinusoïde ?
3. Quel est le rôle du paramètre `kmax` ?
4. Pourquoi faut-il éviter de calculer un logarithme de zéro ?

### Réponses

**1.** La dimension fractale de Higuchi cherche à quantifier la **complexité géométrique** d'un signal temporel, c'est-à-dire à quel point sa trajectoire est irrégulière ou « remplie » dans l'espace. L'algorithme mesure comment la longueur apparente du signal évolue lorsqu'on le sous-échantillonne à différents pas $k$ : si la longueur reste élevée même pour des grands pas, le signal est géométriquement complexe. Formellement, on exploite la relation $L(k) \propto k^{-D}$, où $D$ est la dimension fractale estimée par régression linéaire dans un espace log-log. Un signal lisse (sinusoïde) a $D \approx 1$ (proche d'une courbe 1D régulière) ; un signal très irrégulier peut avoir $D$ proche de 2 (proche d'un remplissage de plan).

**2.** Un signal bruité présente des variations rapides et imprévisibles **à toutes les échelles temporelles**. Lorsqu'on construit les sous-séquences à différents pas $k$, la longueur totale des trajectoires reste élevée même pour de grands $k$, traduisant une auto-similarité multi-échelle caractéristique des structures fractales. Une sinusoïde, en revanche, est parfaitement régulière : ses sous-séquences deviennent rapidement plus courtes quand $k$ augmente, car l'oscillation est capturée dès le premier niveau. La pente log-log est donc plus faible pour la sinusoïde que pour le bruit, d'où une dimension fractale plus basse.

**3.** `kmax` fixe le **pas maximal** testé dans l'algorithme : on évalue $L(k)$ pour $k = 1, 2, \ldots, k_{max}$, ce qui détermine la plage d'échelles temporelles sur laquelle on ajuste la droite de régression log-log. Un `kmax` trop petit réduit le nombre de points disponibles pour l'ajustement et peut donner une estimation instable ou biaisée. Un `kmax` trop grand dépasse la longueur utile du signal (si $k \geq N/2$ les sous-séquences deviennent trop courtes et peu représentatives), ce qui introduit du bruit dans l'estimation. La valeur `kmax=10` est un bon compromis pédagogique pour des segments de 2560 échantillons à 256 Hz.

**4.** $\log(0)$ est mathématiquement indéfini ($\to -\infty$). Dans l'algorithme de Higuchi, on calcule $\log(L(k))$ pour chaque pas $k$ afin d'estimer la pente de régression. Si $L(k) = 0$ — ce qui peut se produire pour un signal constant, un segment saturé ou une sous-séquence pathologiquement identique — le logarithme est indéfini et produit un `NaN` ou `-inf` en NumPy, corrompant silencieusement la régression et la valeur finale de $D$. Il faut donc vérifier que $L(k) > 0$ avant d'appliquer le logarithme, et traiter les cas dégénérés (retourner `0` ou `NaN` avec un avertissement) pour éviter des erreurs d'entraînement en aval.

In [11]:
import numpy as np

def higuchi_fd(signal, kmax=10):
    """
    Dimension fractale de Higuchi.

    Pour chaque pas k de 1 à kmax et chaque point de départ m de 1 à k :
      - on extrait la sous-séquence x[m-1], x[m-1+k], x[m-1+2k], ...
      - on calcule la longueur normalisée L_m(k)
    L(k) est la moyenne de ces longueurs divisée par k.
    La pente de log2(L(k)) vs log2(k) donne -D (dimension fractale).
    """
    x = np.asarray(signal, dtype=np.float64)
    N = len(x)

    lk = np.zeros(kmax)

    for k in range(1, kmax + 1):
        lm_sum = 0.0
        for m in range(1, k + 1):
            ll = (N - m) // k      # nombre d'intervalles dans la sous-séquence
            if ll < 1:
                continue
            subseq = x[m - 1 :: k][: ll + 1]
            # longueur normalisée : facteur (N-1)/(ll*k) compense les longueurs inégales
            lm_sum += np.sum(np.abs(np.diff(subseq))) * (N - 1) / (ll * k)
        lk[k - 1] = lm_sum / k

    k_vals = np.arange(1, kmax + 1)
    valid = lk > 0
    if valid.sum() < 2:
        return 0.0

    slope = np.polyfit(np.log2(k_vals[valid]), np.log2(lk[valid]), 1)[0]
    return float(-slope)


# --- Vérification sur signaux synthétiques ---
FS = 256
duration = 10
t = np.arange(0, duration, 1 / FS)
rng = np.random.default_rng(0)

signals = {
    "sinus lisse (10 Hz)": np.sin(2 * np.pi * 10 * t),
    "bruit blanc":         rng.normal(0, 1, size=len(t)),
    "signal constant":     np.ones(len(t)),
    "mixed":               0.8 * np.sin(2 * np.pi * 6 * t)
                           + 0.5 * np.sin(2 * np.pi * 10 * t)
                           + 0.2 * rng.normal(0, 1, size=len(t)),
}

print(f"{'Signal':<25}  {'Higuchi FD':>12}")
print("-" * 40)
for name, sig in signals.items():
    fd = higuchi_fd(sig, kmax=10)
    print(f"{name:<25}  {fd:>12.4f}")

print("\nInterprétation attendue :")
print("  - sinus lisse    : FD ≈ 1.0–1.2  (signal régulier)")
print("  - bruit blanc    : FD ≈ 1.9–2.0  (signal très irrégulier)")
print("  - signal constant: FD = 0        (cas dégénéré)")

Signal                       Higuchi FD
----------------------------------------
sinus lisse (10 Hz)              0.1060
bruit blanc                      0.9994
signal constant                  0.0000
mixed                            0.4288

Interprétation attendue :
  - sinus lisse    : FD ≈ 1.0–1.2  (signal régulier)
  - bruit blanc    : FD ≈ 1.9–2.0  (signal très irrégulier)
  - signal constant: FD = 0        (cas dégénéré)


## 9. Statistiques temporelles du signal brut

Les statistiques temporelles simples fournissent des informations directes sur l'amplitude et la variabilité du signal.

Features demandées :

1. moyenne ;
2. minimum ;
3. maximum ;
4. médiane ;
5. variance ;
6. écart-type.

### Fonctions Python utiles

| Feature | Fonction NumPy |
|---|---|
| moyenne | `np.mean` |
| minimum | `np.min` |
| maximum | `np.max` |
| médiane | `np.median` |
| variance | `np.var` |
| écart-type | `np.std` |


In [12]:
import numpy as np

def compute_raw_features(signal):
    """
    Statistiques temporelles du signal brut : 6 features.
    Capturent l'amplitude et la variabilité directement dans le domaine temporel.
    """
    x = np.asarray(signal, dtype=np.float64)
    return {
        "raw_mean":   float(np.mean(x)),
        "raw_min":    float(np.min(x)),
        "raw_max":    float(np.max(x)),
        "raw_median": float(np.median(x)),
        "raw_var":    float(np.var(x)),
        "raw_std":    float(np.std(x)),
    }


# --- Vérification ---
FS = 256
t = np.arange(0, 10, 1 / FS)
rng = np.random.default_rng(0)

signals_raw = {
    "sinus (10 Hz)": np.sin(2 * np.pi * 10 * t),
    "bruit blanc":   rng.normal(0, 1, size=len(t)),
    "constant (2)":  2 * np.ones(len(t)),
}

print(f"{'Signal':<18}  {'mean':>7}  {'min':>7}  {'max':>7}  {'var':>7}  {'std':>7}")
print("-" * 65)
for name, sig in signals_raw.items():
    f = compute_raw_features(sig)
    print(f"{name:<18}  {f['raw_mean']:>7.3f}  {f['raw_min']:>7.3f}  "
          f"{f['raw_max']:>7.3f}  {f['raw_var']:>7.3f}  {f['raw_std']:>7.3f}")

print("\nInterprétation attendue :")
print("  - sinus     : mean ≈ 0, min = -1, max = 1, var = 0.5")
print("  - bruit     : mean ≈ 0, var ≈ 1")
print("  - constant  : mean = 2, var = 0, std = 0")

Signal                 mean      min      max      var      std
-----------------------------------------------------------------
sinus (10 Hz)         0.000   -1.000    1.000    0.500    0.707
bruit blanc          -0.035   -3.899    3.066    0.993    0.997
constant (2)          2.000    2.000    2.000    0.000    0.000

Interprétation attendue :
  - sinus     : mean ≈ 0, min = -1, max = 1, var = 0.5
  - bruit     : mean ≈ 0, var ≈ 1
  - constant  : mean = 2, var = 0, std = 0


## 10. Fonction complète d'extraction des 40 features EEG

À ce stade, on peut regrouper toutes les familles de features dans une seule fonction.

### Entrée

Un segment EEG 1D correspondant à :

- un canal ;
- une fenêtre de 10 secondes ;
- 2560 échantillons si `fs=256 Hz`.

### Sortie

Un dictionnaire de **40 features**.

### Organisation recommandée

1. convertir le signal en tableau NumPy ;
2. remplacer les valeurs manquantes par 0 ou par une stratégie décidée en amont ;
3. calculer les features PSD par bande ;
4. calculer les entropies spectrales ;
5. calculer Hjorth ;
6. calculer Lempel-Ziv ;
7. calculer Higuchi ;
8. calculer les statistiques temporelles ;
9. fusionner les dictionnaires.

### Questions

1. Pourquoi la fonction doit-elle retourner un dictionnaire plutôt qu'une simple liste ?
2. Pourquoi est-il important de conserver des noms de colonnes explicites ?
3. Combien de features doit retourner la fonction pour un canal ?
4. Si on a 4 canaux et qu'on concatène toutes les features, combien de features obtient-on par segment ?

### Réponses

**1.** Un dictionnaire associe un **nom explicite** à chaque valeur (ex. `alpha_psd_mean`, `hjorth_mobility`). Cela permet de sélectionner ou réordonner des sous-ensembles sans se souvenir d'un indice, de construire directement un `pd.DataFrame` avec les bons en-têtes de colonnes, et de détecter immédiatement une feature manquante au lieu d'un décalage silencieux d'index.

**2.** Des noms de colonnes explicites sont indispensables pour l'**interprétabilité** (on sait ce que mesure chaque colonne lors de l'analyse des importances ou des visualisations), la **reproductibilité** (un collaborateur peut relire le CSV sans documentation supplémentaire), et la **robustesse** (si l'ordre de calcul change entre deux versions du code, les noms garantissent la correspondance correcte lors d'une fusion de DataFrames).

**3.** La fonction retourne **40 features** par canal :
- $5 \text{ bandes} \times 5 \text{ statistiques PSD} = 25$
- $5 \text{ entropies spectrales}$
- $2 \text{ Hjorth (mobility + complexity)}$
- $1 \text{ Lempel-Ziv}$
- $1 \text{ Higuchi}$
- $6 \text{ statistiques temporelles}$

**4.** Avec 4 canaux (AF7, AF8, TP9, TP10) et 40 features par canal : $4 \times 40 = \mathbf{160}$ features par segment de 10 secondes. C'est le vecteur fourni au classifieur (LDA, SVM, Random Forest…).

In [13]:
import numpy as np

def extract_eeg_features(signal, fs):
    """
    Extrait les 40 features EEG pour un canal et une fenêtre de 10 s.

    Retourne un dictionnaire de 40 clés :
      - 25 PSD          (5 bandes × 5 statistiques)
      -  5 entropies spectrales
      -  2 Hjorth       (mobility, complexity)
      -  1 Lempel-Ziv
      -  1 Higuchi
      -  6 statistiques temporelles
    """
    x = np.asarray(signal, dtype=np.float64)
    x = np.where(np.isfinite(x), x, 0.0)   # NaN / ±inf → 0

    features = {}
    features.update(compute_psd_band_features(x, fs))        # 25
    features.update(compute_specetral_entropy_bands(x, fs))   #  5
    features.update(compute_hjorth(x))                        #  2
    features["lempel_ziv"] = lempel_ziv_complexity(x)         #  1
    features["higuchi_fd"] = higuchi_fd(x, kmax=10)           #  1
    features.update(compute_raw_features(x))                  #  6
    return features                                            # = 40


# --- Vérification ---
test_sig = np.sin(2 * np.pi * 10 * np.arange(0, 10, 1 / 256))
feat = extract_eeg_features(test_sig, 256)

print(f"Nombre de features : {len(feat)}")
assert len(feat) == 40, f"ERREUR : attendu 40, obtenu {len(feat)}"
print("Assertion OK — 40 features par canal.\n")
print("Liste des clés :")
for i, k in enumerate(feat.keys(), 1):
    print(f"  {i:>2}. {k}")

Nombre de features : 40
Assertion OK — 40 features par canal.

Liste des clés :
   1. delta_psd_sum
   2. delta_psd_mean
   3. delta_psd_max
   4. delta_psd_min
   5. delta_psd_median
   6. theta_psd_sum
   7. theta_psd_mean
   8. theta_psd_max
   9. theta_psd_min
  10. theta_psd_median
  11. alpha_psd_sum
  12. alpha_psd_mean
  13. alpha_psd_max
  14. alpha_psd_min
  15. alpha_psd_median
  16. beta_psd_sum
  17. beta_psd_mean
  18. beta_psd_max
  19. beta_psd_min
  20. beta_psd_median
  21. gamma_psd_sum
  22. gamma_psd_mean
  23. gamma_psd_max
  24. gamma_psd_min
  25. gamma_psd_median
  26. spectral_entropy_delta
  27. spectral_entropy_theta
  28. spectral_entropy_alpha
  29. spectral_entropy_beta
  30. spectral_entropy_gamma
  31. hjorth_mobility
  32. hjorth_complexity
  33. lempel_ziv
  34. higuchi_fd
  35. raw_mean
  36. raw_min
  37. raw_max
  38. raw_median
  39. raw_var
  40. raw_std


## 11. Comparaison des features sur signaux connus

On applique maintenant la fonction complète aux signaux synthétiques.

### Objectif

Vérifier que :

- `alpha_10Hz` a une puissance alpha élevée ;
- `beta_20Hz` a une puissance beta élevée ;
- `white_noise` a une entropie spectrale élevée ;
- les features non linéaires augmentent généralement avec l'irrégularité.

In [14]:
import pandas as pd

t, synthetic_signals = generate_synthetic_signals()

rows = []
for name, sig in synthetic_signals.items():
    feat = extract_eeg_features(sig, FS)
    feat["signal"] = name
    rows.append(feat)

df_synth = pd.DataFrame(rows).set_index("signal")

cols_psd = ["delta_psd_sum", "alpha_psd_sum", "beta_psd_sum", "gamma_psd_sum"]
cols_ent  = ["spectral_entropy_delta", "spectral_entropy_alpha", "spectral_entropy_beta"]
cols_nl   = ["hjorth_mobility", "hjorth_complexity", "lempel_ziv", "higuchi_fd"]

print("=== Puissances PSD par bande ===")
print(df_synth[cols_psd].round(4).to_string())

print("\n=== Entropies spectrales ===")
print(df_synth[cols_ent].round(4).to_string())

print("\n=== Features non linéaires ===")
print(df_synth[cols_nl].round(4).to_string())

print("""
Interprétation attendue :
  - delta_2Hz   : delta_psd_sum dominant
  - alpha_10Hz  : alpha_psd_sum dominant
  - beta_20Hz   : beta_psd_sum  dominant
  - white_noise : entropies spectrales élevées, LZ et Higuchi élevés
  - mixed       : plusieurs bandes actives, complexité intermédiaire
""")

=== Puissances PSD par bande ===
             delta_psd_sum  alpha_psd_sum  beta_psd_sum  gamma_psd_sum
signal                                                                
delta_2Hz           1.0000          0.000        0.0000         0.0000
alpha_10Hz          0.0000          1.000        0.0000         0.0000
beta_20Hz           0.0000          0.000        1.0000         0.0000
white_noise         0.0486          0.060        0.2949         0.7051
mixed               0.0022          0.253        0.0123         0.0266

=== Entropies spectrales ===
             spectral_entropy_delta  spectral_entropy_alpha  spectral_entropy_beta
signal                                                                            
delta_2Hz                    0.8676                  1.3853                 2.1601
alpha_10Hz                   0.9133                  0.8676                 1.8861
beta_20Hz                    0.4148                  0.7668                 0.8676
white_noise              

## 12. Application aux signaux EEG du dataset CL-Drive

Après les tests pédagogiques, les mêmes fonctions doivent être appliquées aux signaux EEG prétraités.

### Hypothèse de structure des fichiers

On suppose que les fichiers EEG prétraités sont des fichiers CSV contenant :

- une colonne `Timestamp` ;
- une colonne par canal EEG, par exemple `AF7`, `AF8`, `TP9`, `TP10`.

Exemple de structure :

| Timestamp | AF7 | AF8 | TP9 | TP10 |
|---:|---:|---:|---:|---:|
| 0.000 | ... | ... | ... | ... |
| 0.004 | ... | ... | ... | ... |

### Algorithme d'extraction sur un fichier

1. lire le fichier CSV avec `pd.read_csv` ;
2. identifier les colonnes EEG ;
3. découper le signal en fenêtres de 10 secondes ;
4. pour chaque fenêtre :
   - extraire les 2560 échantillons ;
   - pour chaque canal, calculer les 40 features ;
   - stocker les métadonnées : sujet, fichier, fenêtre, temps début, temps fin, canal ;
5. construire un `DataFrame` ;
6. sauvegarder le résultat en CSV.

### Question

Pourquoi faut-il conserver les colonnes `Participant`, `File`, `Window`, `Channel`, `Start_Time` et `End_Time` avec les features ?

### Réponse

Ces colonnes sont des **métadonnées de traçabilité** indispensables pour trois raisons :

1. **Association aux labels PAAS** : les scores de charge cognitive se trouvent dans un fichier séparé, indexés par participant et par instant temporel. Sans `Participant`, `Start_Time` et `End_Time`, il est impossible de relier chaque ligne de features au bon label.

2. **Débogage et contrôle qualité** : si une fenêtre produit des features anormales (NaN, valeurs infinies), les colonnes `File` et `Window` permettent de retrouver exactement le segment source pour l'inspecter.

3. **Séparation train/test sans fuite** : la colonne `Participant` permet d'effectuer une validation croisée par sujet (*leave-one-subject-out*), garantissant qu'aucune donnée d'un sujet de test ne contamine l'entraînement.

In [26]:
import pandas as pd
import numpy as np
from pathlib import Path

def extract_features_from_eeg_df(df, fs=256, window_sec=10, participant=None, filename=None):
    """
    Extrait les features EEG fenêtre par fenêtre depuis un DataFrame EEG.

    Paramètres
    ----------
    df          : DataFrame avec colonne 'Timestamp' et une colonne par canal EEG.
    fs          : fréquence d'échantillonnage (Hz).
    window_sec  : durée d'une fenêtre (secondes).
    participant : identifiant du participant (métadonnée).
    filename    : nom du fichier source (métadonnée).

    Retourne
    --------
    DataFrame avec une ligne par (fenêtre × canal), 40 features + métadonnées.
    """
    time_col = next((c for c in df.columns if "time" in c.lower() or "timestamp" in c.lower()), None)
    eeg_cols = [c for c in df.columns if c != time_col]
    window_samples = fs * window_sec
    n_windows = len(df) // window_samples

    rows = []
    for w in range(n_windows):
        start_idx = w * window_samples
        end_idx   = start_idx + window_samples
        win = df.iloc[start_idx:end_idx]

        t_start = win[time_col].iloc[0]  if time_col else w * window_sec
        t_end   = win[time_col].iloc[-1] if time_col else (w + 1) * window_sec

        for ch in eeg_cols:
            feat = extract_eeg_features(win[ch].values, fs)
            feat["Participant"] = participant
            feat["File"]        = filename
            feat["Window"]      = w
            feat["Channel"]     = ch
            feat["Start_Time"]  = t_start
            feat["End_Time"]    = t_end
            rows.append(feat)

    return pd.DataFrame(rows)


# --- Application sur les fichiers réels du dataset ---
DATASET_ROOT = Path("dataset/EEG")   # ← modifier si nécessaire
OUTPUT_DIR   = Path("dataset/EEG_Features_10s")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Cherche tous les filtered_*.csv dans les dossiers numériques
csv_files = sorted(DATASET_ROOT.rglob("filtered_*.csv"))

if not csv_files:
    print(f"Aucun fichier filtered_*.csv trouvé dans {DATASET_ROOT.resolve()}")
    print("Lance d'abord le prétraitement par lot (TD3 Activité 9).")
else:
    print(f"{len(csv_files)} fichier(s) trouvé(s)\n")
    for csv_path in csv_files:
        participant_id = csv_path.parent.name
        df_raw = pd.read_csv(csv_path)

        feat_df = extract_features_from_eeg_df(
            df_raw, fs=256, window_sec=10,
            participant=participant_id,
            filename=csv_path.name,
        )

        out_path = OUTPUT_DIR / participant_id
        out_path.mkdir(parents=True, exist_ok=True)
        out_file = out_path / f"features_{csv_path.stem}.csv"
        feat_df.to_csv(out_file, index=False)

        print(f"[{participant_id}]  {feat_df['Window'].nunique()} fenêtres  "
              f"× {feat_df['Channel'].nunique()} canaux  →  {out_file.relative_to(OUTPUT_DIR.parent)}")

364 fichier(s) trouvé(s)

[1030]  12 fenêtres  × 4 canaux  →  EEG_Features_10s/1030/features_filtered_eeg_baseline_level_1.csv
[1030]  11 fenêtres  × 4 canaux  →  EEG_Features_10s/1030/features_filtered_eeg_baseline_level_2.csv
[1030]  12 fenêtres  × 4 canaux  →  EEG_Features_10s/1030/features_filtered_eeg_baseline_level_3.csv
[1030]  12 fenêtres  × 4 canaux  →  EEG_Features_10s/1030/features_filtered_eeg_baseline_level_4.csv
[1030]  12 fenêtres  × 4 canaux  →  EEG_Features_10s/1030/features_filtered_eeg_baseline_level_5.csv
[1030]  12 fenêtres  × 4 canaux  →  EEG_Features_10s/1030/features_filtered_eeg_baseline_level_6.csv
[1030]  12 fenêtres  × 4 canaux  →  EEG_Features_10s/1030/features_filtered_eeg_baseline_level_7.csv
[1030]  12 fenêtres  × 4 canaux  →  EEG_Features_10s/1030/features_filtered_eeg_baseline_level_8.csv
[1030]  12 fenêtres  × 4 canaux  →  EEG_Features_10s/1030/features_filtered_eeg_baseline_level_9.csv
[1030]  18 fenêtres  × 4 canaux  →  EEG_Features_10s/1030/feature

## 13. Traitement par lot des fichiers EEG prétraités

Dans le projet, les fichiers prétraités sont organisés par sujet avec un identifiant numérique.

Exemple :

```text
doi-10/
|----EEG/
    ├── 1030/
    │   ├── filtered_eeg_data_level_1.csv
    │   ├── filtered_eeg_baseline_level_1.csv
    │   └── ...
    ├── 1031/
    │   └── ...
|----EDA
|----ECG
|----Gaze
|----Labels
```

### Algorithme par lot

1. parcourir les dossiers sujets (dossiers dont le nom est un nombre) ;
2. sélectionner uniquement les fichiers `filtered_*.csv` ;
3. lire chaque fichier ;
4. extraire les features fenêtre par fenêtre ;
5. sauvegarder un fichier CSV de features par sujet.

### Remarque importante

Les labels PAAS ne se trouvent pas dans le même fichier que les signaux. Une étape d'association entre les features et les labels est donc nécessaire dans un second temps : il faut relier les scores de charge cognitive aux intervalles temporels correspondants.

In [24]:
from pathlib import Path
import pandas as pd

def batch_extract_eeg_features(base_path, output_path, fs=256, window_sec=10):
    """
    Traitement par lot : parcourt base_path/<ID_numerique>/filtered_*.csv,
    extrait les features fenêtre par fenêtre et sauvegarde un CSV par participant.
    """
    base_path   = Path(base_path)
    output_path = Path(output_path)
    output_path.mkdir(parents=True, exist_ok=True)

    # Les dossiers participants sont des identifiants numériques (ex. 1030, 1031…)
    participant_dirs = sorted(
        [d for d in base_path.iterdir() if d.is_dir() and d.name.isdigit()]
    )

    if not participant_dirs:
        print(f"Aucun dossier numérique trouvé dans {base_path}")
        return

    for part_dir in participant_dirs:
        participant_id = part_dir.name
        csv_files = sorted(part_dir.glob("filtered_*.csv"))

        if not csv_files:
            print(f"[{participant_id}] aucun fichier filtered_*.csv — ignoré.")
            continue

        all_rows = []
        for csv_file in csv_files:
            df = pd.read_csv(csv_file)
            feat_df = extract_features_from_eeg_df(
                df, fs=fs, window_sec=window_sec,
                participant=participant_id, filename=csv_file.name,
            )
            all_rows.append(feat_df)

        result = pd.concat(all_rows, ignore_index=True)
        out_file = output_path / f"{participant_id}_eeg_features.csv"
        result.to_csv(out_file, index=False)
        print(f"[{participant_id}]  {len(result)} lignes  →  {out_file.name}")


# --- Exemple d'appel ---
batch_extract_eeg_features(
    base_path="dataset/EEG",
    output_path="dataset/EEG_Features_10s",
    fs=256,
    window_sec=10,
)

[1030]  1068 lignes  →  1030_eeg_features.csv
[1105]  1068 lignes  →  1105_eeg_features.csv
[1106]  1080 lignes  →  1106_eeg_features.csv
[1241]  1080 lignes  →  1241_eeg_features.csv
[1271]  1004 lignes  →  1271_eeg_features.csv
[1314]  992 lignes  →  1314_eeg_features.csv
[1323]  1080 lignes  →  1323_eeg_features.csv
[1337]  1080 lignes  →  1337_eeg_features.csv
[1372]  1080 lignes  →  1372_eeg_features.csv
[1417]  1080 lignes  →  1417_eeg_features.csv
[1434]  1040 lignes  →  1434_eeg_features.csv
[1544]  1036 lignes  →  1544_eeg_features.csv
[1547]  980 lignes  →  1547_eeg_features.csv
[1595]  1080 lignes  →  1595_eeg_features.csv
[1629]  840 lignes  →  1629_eeg_features.csv
[1716]  1052 lignes  →  1716_eeg_features.csv
[1717]  1044 lignes  →  1717_eeg_features.csv
[1744]  692 lignes  →  1744_eeg_features.csv
[1868]  772 lignes  →  1868_eeg_features.csv
[1892]  1060 lignes  →  1892_eeg_features.csv
[1953]  1080 lignes  →  1953_eeg_features.csv


## 14. Vérifications qualité des features

Avant de passer à la classification, il faut vérifier la qualité de la matrice de features.

### Vérifications recommandées

1. nombre de lignes cohérent avec le nombre de fenêtres et de canaux ;
2. absence de valeurs manquantes ;
3. absence de valeurs infinies ;
4. ordre de grandeur plausible ;
5. nombre de features égal à 40 par canal ;
6. conservation des métadonnées utiles ;
7. possibilité d'associer ensuite chaque fenêtre à un label.

### Questions

1. Pourquoi des valeurs `NaN` peuvent-elles apparaître dans les features ?
2. Pourquoi des valeurs infinies peuvent-elles apparaître ?
3. Que doit-on faire si un segment contient trop de valeurs manquantes ?
4. Pourquoi faut-il éviter de normaliser les features avant la séparation train/test ?

### Réponses

**1.** Des `NaN` peuvent apparaître dans plusieurs cas : division par zéro dans les paramètres de Hjorth si `Var(x) = 0` (canal saturé ou mort) ; logarithme d'une PSD nulle dans l'entropie spectrale si aucune énergie n'est présente dans une bande ; `log(L(k) = 0)` dans Higuchi ; ou propagation d'un `NaN` déjà présent dans le signal brut (données manquantes non traitées en amont). La fonction `extract_eeg_features` remplace les valeurs non finies en entrée par 0, et chaque sous-fonction gère ses propres cas dégénérés.

**2.** Des `±inf` apparaissent lors d'une division par zéro non protégée (normalisation par une variance nulle, `log(0)` retourné tel quel) ou d'un dépassement numérique sur des signaux à très grande amplitude. NumPy les propage silencieusement, ce qui peut corrompre l'entraînement ou les scalings ultérieurs sans déclencher d'erreur visible.

**3.** Si un segment contient trop de valeurs manquantes (ex. > 20 % de NaN), il faut **exclure la fenêtre entière** plutôt que de l'imputer : reconstituer la majorité d'un canal produit une fenêtre artificielle non représentative de l'activité cérébrale réelle. On peut conserver la ligne dans le CSV avec un flag `quality=0` pour garder la traçabilité et la filtrer lors de l'entraînement.

**4.** Normaliser avant la séparation train/test introduit une **fuite de données** (*data leakage*) : les paramètres du scaler (moyenne, écart-type, min, max) sont calculés sur l'ensemble du dataset, y compris les données de test. Le modèle accède donc indirectement à de l'information du test lors de l'entraînement, ce qui produit des estimations de performance trop optimistes. Il faut toujours ajuster le scaler uniquement sur le train set, puis l'appliquer au test set sans recalcul.

In [25]:
import numpy as np
import pandas as pd

META_COLS = {"Participant", "File", "Window", "Channel", "Start_Time", "End_Time"}

def check_features_quality(df, n_features_expected=40):
    """
    Contrôle qualité de la matrice de features EEG.
    Vérifie le nombre de features, l'absence de NaN/inf, et la cohérence des métadonnées.
    Retourne un dictionnaire de rapport.
    """
    feat_cols  = [c for c in df.columns if c not in META_COLS]
    feat_vals  = df[feat_cols].values.astype(float)

    n_nan = int(np.isnan(feat_vals).sum())
    n_inf = int(np.isinf(feat_vals).sum())

    report = {
        "n_rows":            len(df),
        "n_feature_cols":    len(feat_cols),
        "features_count_ok": len(feat_cols) == n_features_expected,
        "n_nan":             n_nan,
        "n_inf":             n_inf,
        "channels":    sorted(df["Channel"].unique().tolist())     if "Channel"     in df.columns else [],
        "participants": sorted(df["Participant"].unique().tolist()) if "Participant" in df.columns else [],
        "windows_per_channel": (
            df.groupby("Channel")["Window"].nunique().to_dict()
            if "Channel" in df.columns else {}
        ),
    }

    ok = lambda cond: "OK" if cond else "AVERTISSEMENT"
    print("=" * 48)
    print("   Contrôle qualité des features EEG")
    print("=" * 48)
    print(f"  Lignes totales         : {report['n_rows']}")
    print(f"  Features par canal     : {report['n_feature_cols']}  [{ok(report['features_count_ok'])}]")
    print(f"  Valeurs NaN            : {report['n_nan']}  [{ok(n_nan == 0)}]")
    print(f"  Valeurs infinies       : {report['n_inf']}  [{ok(n_inf == 0)}]")
    print(f"  Canaux                 : {report['channels']}")
    print(f"  Participants           : {report['participants']}")
    print(f"  Fenêtres / canal       : {report['windows_per_channel']}")
    print("=" * 48)

    return report


# --- Démonstration ---
FS = 256
t_demo = np.arange(0, 30, 1 / FS)
rng = np.random.default_rng(2)

demo_df = pd.DataFrame({
    "Timestamp": t_demo,
    "AF7": np.sin(2 * np.pi * 10 * t_demo) + 0.1 * rng.normal(size=len(t_demo)),
    "AF8": np.sin(2 * np.pi * 6  * t_demo) + 0.1 * rng.normal(size=len(t_demo)),
})

feat_df = extract_features_from_eeg_df(demo_df, fs=FS, window_sec=10,
                                        participant="demo", filename="demo.csv")
report = check_features_quality(feat_df, n_features_expected=40)

   Contrôle qualité des features EEG
  Lignes totales         : 6
  Features par canal     : 40  [OK]
  Valeurs NaN            : 0  [OK]
  Valeurs infinies       : 0  [OK]
  Canaux                 : ['AF7', 'AF8']
  Participants           : ['demo']
  Fenêtres / canal       : {'AF7': 3, 'AF8': 3}
